## Оценка качества извлечения

Идея бенчмарка состоит в измерении качества извлечения документов по описанию (docstring) типов, методов, функций и т.п., содержащихся в них, в зависимости от используемого алгортима сегментации исходных текстов. Сами docstring при этом удаляются из исходного кода. Помимо представленного алгоритма на основе AST, используются стандартные функции сегментации из LlamaIndex:

- `SentenceSplitter` - базовый алгоритм разбиения текстов на фиксированные сегменты.
- `CodeSplitter` - алгоритм, разбивающий код только по границам блоков кода.
- `BM25` - алгоритм Okapi BM25 (без сегментации, поиск осуществляется по всему документу).

In [ ]:
import torch
import datasets
import pandas as pd
import numpy as np

from llama_index.core import VectorStoreIndex, Settings, StorageContext, load_index_from_storage, QueryBundle
from llama_index.core.storage.docstore import SimpleDocumentStore
from llama_index.core.storage.index_store import SimpleIndexStore
from llama_index.core.vector_stores import SimpleVectorStore
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import Document
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from tqdm.std import tqdm
from beir.retrieval.evaluation import EvaluateRetrieval
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.query_engine.retriever_query_engine import RetrieverQueryEngine
from llama_index.core.node_parser import CodeSplitter
from chunking import SourceCodeNodeParser

np.random.seed(42)

Для оценки качества извлечения данных был собран [набор данных](https://huggingface.co/datasets/kmvi/code-ir-dataset), состоящий из трех блоков:

- `docs` - файлы исходного кода (документы) с удаленными из них docstring:
    - `doc_id` - идентификатор документа
    - `file` - имя файла
    - `content` - содержимое файла (с удаленными docstring's)
- `queries` - запросы к документам. В качестве запросов выступают docstring
    - `query_id` - идентификатор запроса
    - `query` - текст запроса
    - `type` - тип элемента кода, к которому относился docstring (тип, либо метод/функция)
- `qrels` - корректные пары запрос/документ
    - `query_id` - идентификатор запроса
    - `doc_id` - идентификатор документа

In [2]:
docs_ds = datasets.load_dataset("kmvi/code-ir-dataset", name="docs")
queries_ds = datasets.load_dataset("kmvi/code-ir-dataset", name="queries")
qrels_ds = datasets.load_dataset("kmvi/code-ir-dataset", name="qrels")

doc_ids = { row['file']: row['doc_id'] for row in docs_ds['train'] }
qrels = qrels_ds['train'].to_pandas().groupby('query_id').doc_id.apply(list).apply(lambda x: {k: 1 for k in x})

Подготавливаем исходные тексты для построения индекса.

In [3]:
docs = []
for row in docs_ds['train']:
    with open(row['file'], 'rt', encoding='utf-8') as f:
        text = f.read().replace('\r\n', '\n')

    doc = Document(id_=row['doc_id'], text=text, metadata={'file_name': row['file']})
    docs.append(doc)

Загрузка модели эмбеддингов:

In [ ]:
Settings.embed_model = HuggingFaceEmbedding(
    model_name="Qwen/Qwen3-Embedding-0.6B",
    device="cuda",
    normalize=True,
    model_kwargs={'dtype': torch.bfloat16},
)

Settings.llm = None

LLM is explicitly disabled. Using MockLLM.


Построение индекса с использованием SentenceSplitter.

In [5]:
storage_context = StorageContext.from_defaults(
    docstore=SimpleDocumentStore(),
    vector_store=SimpleVectorStore(),
    index_store=SimpleIndexStore(),
)

splitter = SentenceSplitter(chunk_size=256)
index = VectorStoreIndex.from_documents(docs, storage_context, transformations=[splitter], show_progress=True, store_nodes_override=True)
storage_context.persist('context')

Parsing nodes:   0%|          | 0/175 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/673 [00:00<?, ?it/s]

Построение индекса с использованием CodeSplitter:

In [6]:
storage_context = StorageContext.from_defaults(
    docstore=SimpleDocumentStore(),
    vector_store=SimpleVectorStore(),
    index_store=SimpleIndexStore(),
)

splitter = CodeSplitter('csharp', max_chars=256)
index = VectorStoreIndex.from_documents(docs, storage_context, transformations=[splitter], show_progress=True, store_nodes_override=True)
storage_context.persist('context_codesplitter')

Parsing nodes:   0%|          | 0/175 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/276 [00:00<?, ?it/s]

Построение индекса с использованием нового алгоритма на основе AST:

In [7]:
storage_context = StorageContext.from_defaults(
    docstore=SimpleDocumentStore(),
    vector_store=SimpleVectorStore(),
    index_store=SimpleIndexStore(),
)

parser = SourceCodeNodeParser(chunk_size=256)
nodes = parser(docs)
index = VectorStoreIndex(nodes, storage_context=storage_context, show_progress=True, store_nodes_override=True)
storage_context.persist('context_my')

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/969 [00:00<?, ?it/s]

## Вычисление метрик

Для расчета метрик используется функционал из библиотеки `beir`.

In [ ]:
retr = EvaluateRetrieval()
top_k = 10 # извлекаем 10 документов

In [9]:
storename = 'context'
docstore = SimpleDocumentStore.from_persist_dir(storename)
vector_store = SimpleVectorStore.from_persist_dir(storename)
index_store = SimpleIndexStore.from_persist_dir(storename)
storage_context = StorageContext.from_defaults(
    docstore=docstore,
    vector_store=vector_store,
    index_store=index_store,
)
index = load_index_from_storage(storage_context)

In [10]:
engine = index.as_query_engine(similarity_top_k=top_k)
results = dict()
for row in tqdm(queries_ds['train']):
    data = engine.retrieve(QueryBundle(row['query']))
    results[row['query_id']] = { doc_ids[node.metadata['file_name']]: node.score for node in data}

metrics = retr.evaluate(qrels, results, k_values=[3,5,10])

100%|██████████| 1030/1030 [06:02<00:00,  2.85it/s]


In [11]:
storename = 'context_codesplitter'
docstore = SimpleDocumentStore.from_persist_dir(storename)
vector_store = SimpleVectorStore.from_persist_dir(storename)
index_store = SimpleIndexStore.from_persist_dir(storename)
storage_context = StorageContext.from_defaults(
    docstore=docstore,
    vector_store=vector_store,
    index_store=index_store,
)
index = load_index_from_storage(storage_context)

In [12]:
engine = index.as_query_engine(similarity_top_k=top_k)
results_codesplit = dict()
for row in tqdm(queries_ds['train']):
    data = engine.retrieve(QueryBundle(row['query']))
    results_codesplit[row['query_id']] = { doc_ids[node.metadata['file_name']]: node.score for node in data}

metrics_codesplit = retr.evaluate(qrels, results_codesplit, k_values=[3,5,10])

100%|██████████| 1030/1030 [08:03<00:00,  2.13it/s]


In [13]:
storename = 'context_my'
docstore = SimpleDocumentStore.from_persist_dir(storename)
vector_store = SimpleVectorStore.from_persist_dir(storename)
index_store = SimpleIndexStore.from_persist_dir(storename)
storage_context = StorageContext.from_defaults(
    docstore=docstore,
    vector_store=vector_store,
    index_store=index_store,
)
index = load_index_from_storage(storage_context)

In [14]:
engine = index.as_query_engine(similarity_top_k=top_k)
results_ast = dict()
for row in tqdm(queries_ds['train']):
    data = engine.retrieve(QueryBundle(row['query']))
    results_ast[row['query_id']] = { doc_ids[node.metadata['file_name']]: node.score for node in data}

metrics_ast = retr.evaluate(qrels, results_ast, k_values=[3,5,10])

100%|██████████| 1030/1030 [09:35<00:00,  1.79it/s]


In [15]:
docids = list(doc_ids.values())
results_random = dict()
for row in tqdm(queries_ds['train']):
    results_random[row['query_id']] = {k: 1.0 / top_k for k in np.random.choice(docs_ds['train']['doc_id'], top_k)}

metrics_random = retr.evaluate(qrels, results_random, k_values=[3,5,10])

100%|██████████| 1030/1030 [00:00<00:00, 5916.78it/s]


In [ ]:
bm25 = BM25Retriever.from_defaults(nodes=docs, similarity_top_k=top_k)
engine = RetrieverQueryEngine.from_args(bm25, llm=Settings.llm)
results_bm25 = dict()
for row in tqdm(queries_ds['train']):
    data = engine.retrieve(QueryBundle(row['query']))
    results_bm25[row['query_id']] = { doc_ids[node.metadata['file_name']]: node.score for node in data}

metrics_bm25 = retr.evaluate(qrels, results_bm25, k_values=[3,5,10])

100%|██████████| 1030/1030 [00:01<00:00, 866.67it/s]


Итоговый результат:

In [40]:
rows = []
for m in [metrics_random, metrics_bm25, metrics, metrics_codesplit, metrics_ast]:
    tmp = dict()
    for item in m:
        tmp.update(item)
    rows.append(tmp)

df = pd.DataFrame(rows, index=['Random', 'BM25', 'SentenceSplitter', 'CodeSplitter', 'AST Splitter'])
df = df[df.filter(regex='^NDCG|^P|^Recall').columns]

display(df.style.highlight_max(color='green').format(precision=4))

,NDCG@3,NDCG@5,NDCG@10,Recall@3,Recall@5,Recall@10,P@3,P@5,P@10
Random,0.0141,0.0164,0.0267,0.0185,0.0235,0.0568,0.0068,0.0056,0.0063
BM25,0.2564,0.2933,0.3335,0.3026,0.3907,0.5123,0.1097,0.0854,0.0567
SentenceSplitter,0.5996,0.6172,0.6193,0.6884,0.7281,0.7339,0.2450,0.1573,0.0792
CodeSplitter,0.5889,0.6265,0.6291,0.7203,0.8080,0.8153,0.2573,0.1742,0.0880
AST Splitter,0.6072,0.6360,0.6413,0.7380,0.8056,0.8206,0.2644,0.1740,0.0885
